In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

DIRECT_OPEN_SOURCE_TIMING = PROJECT_DIR / "outputs" / "mllm_notebook_timing.jsonl"
DIRECT_PROPRIETARY_TIMING = BUNDLE_DIR / "proprietary_notebook_results.jsonl"
OPENROUTER_TIMING = PROJECT_DIR / "outputs" / "timing_cost_all_models" / "openrouter_timing.jsonl"


# 05 — Timing and cost

The old cost builder read open-source wall-clock values from cluster log files. That dependency is intentionally gone here. This notebook uses timing recorded around the direct notebook inference call; if no direct timing file exists, it leaves that model out of the table and says why. Proprietary timing comes from API response rows and their reported usage. No estimate is silently substituted.


In [ ]:
import pandas as pd

def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(path, lines=True) if path.is_file() else pd.DataFrame()

def direct_open_source_summary() -> pd.DataFrame:
    rows = read_jsonl(DIRECT_OPEN_SOURCE_TIMING)
    if rows.empty:
        print(f"No direct open-source timing file at {DIRECT_OPEN_SOURCE_TIMING}; run notebook 01 with RECORD_TIMING=True.")
        return pd.DataFrame()
    rows["sec_per_label"] = rows["wall_clock_sec"] / rows["labels"].clip(lower=1)
    return rows.groupby("model_label", as_index=False).agg(labels=("labels", "sum"), wall_clock_sec=("wall_clock_sec", "sum"), sec_per_label=("sec_per_label", "mean")).assign(api_cost_usd=0.0, timing_source="direct_notebook_wall_clock")

def proprietary_summary() -> pd.DataFrame:
    rows = read_jsonl(DIRECT_PROPRIETARY_TIMING)
    if rows.empty and OPENROUTER_TIMING.is_file():
        rows = read_jsonl(OPENROUTER_TIMING)
    if rows.empty:
        print("No proprietary timing rows found; run notebook 02 or point OPENROUTER_TIMING at API timing JSONL.")
        return pd.DataFrame()
    if "generation_elapsed_sec" in rows:
        rows["wall_clock_sec"] = rows["generation_elapsed_sec"].fillna(0) + rows["choice_elapsed_sec"].fillna(0)
        rows["api_cost_usd"] = 0.0
    else:
        rows["wall_clock_sec"] = rows["wall_clock_sec"].astype(float)
        rows["api_cost_usd"] = rows["api_cost_usd"].astype(float)
    return rows.groupby("model_label", as_index=False).agg(labels=("model_label", "size"), wall_clock_sec=("wall_clock_sec", "sum"), api_cost_usd=("api_cost_usd", "sum")).assign(sec_per_label=lambda frame: frame["wall_clock_sec"] / frame["labels"].clip(lower=1), timing_source="direct_api_response")

timing = pd.concat([direct_open_source_summary(), proprietary_summary()], ignore_index=True)
if not timing.empty:
    timing["wall_clock_hr"] = timing["wall_clock_sec"] / 3600
display(timing)


## Plot throughput and API spend

Open-source API cost is zero, but GPU wall-clock is not. Keeping both dimensions visible prevents calling a local model “free” without recording the time it consumed.


In [ ]:
from mu.visv2.figure import Figure

if timing.empty:
    print("No timing rows available to plot.")
else:
    fig = Figure(n_rows=1, n_cols=2, figsize=(12, 4.5), fontfamily="sans-serif", fontsize=11, panel_padding=(0.3, 0.2))
    left, right = fig.panels
    left.p.bar(timing["model_label"].tolist(), timing["sec_per_label"].tolist(), color="#1d4ed8")
    left.add_xticklabel_rotation(65)
    left.set_yaxislabel("seconds per label")
    left.set_title("Direct measured throughput")
    right.p.bar(timing["model_label"].tolist(), timing["api_cost_usd"].tolist(), color="#be123c")
    right.add_xticklabel_rotation(65)
    right.set_yaxislabel("API cost (USD)")
    right.set_title("Observed API spend")
    fig.show()


The existing cache file is not used as a fallback because its open-source timing came from old cluster logs. This explicit omission makes the notebook portable to a new GPU node.
